# Fast Retrieval — Pipeline NHANH (bỏ sinh answer) để leo điểm TRUY HỒI

Chạy **Hybrid Retrieval → Reranking → trích `relevant_docs`/`relevant_articles`**, KHÔNG sinh `answer` (để `answer=""`).
Có **2 chế độ** (đổi ở cell chạy):

| Chế độ | Cờ | Tốc độ | Ý tưởng |
|---|---|---|---|
| **No-LLM** (nhanh nhất) | *(mặc định)* | ~1-2s/câu → 2000 câu ~30-60 phút | Lấy thẳng **top-2 theo rerank** |
| **LLM-select** | `--llm-select --pool-k 6` | ~2-4s/câu → 2000 câu ~1.5-2.5h | Lấy **pool 6** ứng viên rerank rồi **LLM chọn lọc** (kỳ vọng precision cao hơn) |

> ⚠️ Cả hai bỏ `answer` → mất điểm 5 tiêu chí QA; chỉ tối ưu **truy hồi F2**. Dùng notebook đầy đủ `piplne-and-debug` (có LLM sinh answer) cho bản nộp tính điểm QA.
>
> ⚠️ Chế độ **LLM-select CHƯA chắc thắng top-2** — phải **đo lại trên `ground_truth_50`** mới biết. Thử nghiệm trước cho thấy để LLM tự dẫn điều có thể làm giảm F2.
>
> Luồng RIÊNG — không đụng `main.py` / `debug_pipeline.py` / notebook cũ.

In [ ]:
# 1) Cài dependencies (bitsandbytes/accelerate chỉ cần khi dùng --llm-select)
!pip install -q qdrant-client rank_bm25 underthesea sentence-transformers \
    transformers accelerate bitsandbytes python-dotenv tqdm

In [ ]:
# 2) Load secrets Qdrant từ Kaggle
import os
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    os.environ["QDRANT_URL"] = secrets.get_secret("QDRANT_URL")
    os.environ["QDRANT_API_KEY"] = secrets.get_secret("QDRANT_API_KEY")
    print("✅ Secrets loaded from Kaggle")
except Exception as e:
    print("⚠️ Không load được secrets:", e)

In [ ]:
# 3) Clone / pull code từ GitHub — NHÁNH test1 (main giữ nguyên cho mọi người)
import os, sys

GITHUB_USERNAME = "kimmttrung"
REPO_NAME = "legal-rag-vietnam"
BRANCH = "test1"
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
WORKING_DIR = f"/kaggle/working/{REPO_NAME}"

if os.path.exists(WORKING_DIR):
    print(f"🔄 Pull nhánh {BRANCH}...")
    !cd {WORKING_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    print(f"📥 Clone nhánh {BRANCH}...")
    !cd /kaggle/working && git clone -b {BRANCH} {REPO_URL}

if WORKING_DIR not in sys.path:
    sys.path.insert(0, WORKING_DIR)
print(f"✅ Đồng bộ hoàn tất! (branch={BRANCH})")

In [ ]:
# 4) Vào thư mục repo + kiểm tra file cần thiết
import os, sys

WORKING_DIR = "/kaggle/working/legal-rag-vietnam"
os.chdir(WORKING_DIR)
if sys.path[0] != WORKING_DIR:
    sys.path.insert(0, WORKING_DIR)

print("Working directory:", os.getcwd())
for f in ["fast_retrieval.py", "src/reference_extractor.py", "config/settings.py",
          "data/law_corpus_clean.json", "data/law_manifest.json"]:
    print(("✅ " if os.path.exists(f) else "❌ THIẾU: ") + f)

In [ ]:
# 5) Tự tìm file câu hỏi trong Kaggle Dataset (/kaggle/input)
import glob, json

# Nếu auto-detect sai, gán thẳng đường dẫn vào INPUT_FILE bên dưới.
INPUT_FILE = None

if INPUT_FILE is None:
    candidates = glob.glob("/kaggle/input/**/*.json", recursive=True)
    print("Các file JSON tìm thấy trong /kaggle/input:")
    for c in candidates:
        print("  -", c)
    pick = next((c for c in candidates if any(k in c.lower() for k in ["r2ai", "question", "data", "test"])), None)
    INPUT_FILE = pick or (candidates[0] if candidates else None)

assert INPUT_FILE and os.path.exists(INPUT_FILE), "Không tìm thấy file input! Gán thủ công INPUT_FILE."

_raw = json.load(open(INPUT_FILE, encoding="utf-8"))
_qs = _raw if isinstance(_raw, list) else _raw.get("data", _raw.get("questions", []))
NUM_QUESTIONS = len(_qs)
print(f"\n✅ INPUT_FILE = {INPUT_FILE}  ({NUM_QUESTIONS} câu hỏi)")

In [ ]:
# 6) Chạy → /kaggle/working/results.json
#    Thêm  --num-questions 50  để thử nhanh trước khi chạy full.
#
#    CHỌN 1 trong 2 chế độ (bỏ comment dòng tương ứng):
#
#    (A) LLM-SELECT — pool 6 ứng viên rerank, LLM chọn lọc (precision cao hơn, chậm hơn):
print(f"🚀 Fast retrieval {NUM_QUESTIONS} câu — LLM-SELECT (pool-k=6)...\n")
!python fast_retrieval.py --input "{INPUT_FILE}" --output /kaggle/working/results.json --llm-select --pool-k 6

#    (B) NO-LLM — top-2 rerank, nhanh nhất (~30-60 phút cho 2000 câu):
# print(f"🚀 Fast retrieval {NUM_QUESTIONS} câu — NO-LLM (top-2)...\n")
# !python fast_retrieval.py --input "{INPUT_FILE}" --output /kaggle/working/results.json

In [ ]:
# 7) Kiểm tra + đóng gói submission.zip
import json, zipfile, statistics

results = json.load(open("/kaggle/working/results.json", encoding="utf-8"))
na = [len(r["relevant_articles"]) for r in results]
nd = [len(r["relevant_docs"]) for r in results]
empty_art = sum(1 for x in na if x == 0)
print(f"✅ results.json — {len(results)} bản ghi")
print("   id mẫu:", [r['id'] for r in results[:5]], "| kiểu:", type(results[0]['id']).__name__)
print(f"   TB điều/câu={statistics.mean(na):.2f} | TB văn bản/câu={statistics.mean(nd):.2f} | câu rỗng article={empty_art}")

with zipfile.ZipFile("/kaggle/working/submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("/kaggle/working/results.json", arcname="results.json")
print("✅ /kaggle/working/submission.zip (chứa results.json)")

print("\n----- Bản ghi mẫu -----")
print(json.dumps(results[0], ensure_ascii=False, indent=2)[:600])